# BFND step 3: is the image content gain real on the test split?

Run from a fresh runtime with **Runtime → Run all**.

**Before you start, put these in your Google Drive `MyDrive`:**
- `BFND_step3_code.zip` (the current code)
- `BFND-data/features/` with `meta.csv`, `bert.npy`, `clip_img.npy` (from the earlier feature run)

**What it does:**
1. Mounts Drive and checks the files.
2. Unzips the code into the runtime.
3. Runs the new tests.
4. Runs the controls (script 10). This also saves the test probabilities of every control.
5. Runs three paired bootstrap comparisons on the **test** split (script 12).
6. Prints one summary table and copies the results to `MyDrive/BFND-data/exports/<time>/`.

**Notes:**
- A GPU is optional. CPU works; the heads are small.
- The test split was read before (the text vs fusion bootstrap). These three comparisons were fixed in advance (decision D-10). Do not add or change comparisons after seeing the numbers.
- Script output is captured and printed in each cell, so you will see it.

## Cell 1: mount Drive and check the top-level files

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
BFND = f"{DRIVE}/BFND-data"
CODE_ZIP = f"{DRIVE}/BFND_step3_code.zip"

assert os.path.exists(CODE_ZIP), f"Upload BFND_step3_code.zip to MyDrive first. Not found: {CODE_ZIP}"
assert os.path.isdir(f"{BFND}/features"), "BFND-data/features is missing."
print("Drive mounted. Code zip and features folder found.")

Mounted at /content/drive
Drive mounted. Code zip and features folder found.


## Cell 2: install pytest

In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print("pytest ready")

pytest ready


## Cell 3: unzip the code into the runtime

In [3]:
import os, zipfile

EXTRACT = "/content/bfnd_step3"
os.makedirs(EXTRACT, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(EXTRACT)

PROJECT = None
for root, dirs, files in os.walk(EXTRACT):
    if "src" in dirs and "scripts" in dirs and "configs" in dirs:
        PROJECT = root
        break
assert PROJECT is not None, "src/, scripts/ and configs/ not found in the zip"

required = ["scripts/10_confound_controls.py", "scripts/12_bootstrap_ci.py",
            "src/bootstrap.py", "src/confound_controls.py",
            "tests/test_bootstrap.py", "tests/test_confound_controls.py", "tests/test_evidence_eval.py"]
missing = [f for f in required if not os.path.exists(os.path.join(PROJECT, f))]
assert not missing, f"Missing from the zip: {missing}"
assert "--probs-dir" in open(os.path.join(PROJECT, "scripts/12_bootstrap_ci.py"), encoding="utf-8").read(), \
    "script 12 in the zip is too old (no --probs-dir). Rebuild the zip."
assert "CONTROL_DIR / \"probs\"" in open(os.path.join(PROJECT, "scripts/10_confound_controls.py"), encoding="utf-8").read(), \
    "script 10 in the zip is too old (does not save test probabilities). Rebuild the zip."
print("Project folder:", PROJECT)
print("Files found and up to date.")

Project folder: /content/bfnd_step3/BFND_code
Files found and up to date.


## Cell 4: check the feature files

In [4]:
import os
import numpy as np
import pandas as pd

FEATURES = f"{BFND}/features"
for f in ["meta.csv", "bert.npy", "clip_img.npy"]:
    assert os.path.exists(f"{FEATURES}/{f}"), f"Missing: {FEATURES}/{f}"
meta = pd.read_csv(f"{FEATURES}/meta.csv")
assert len(meta) == np.load(f"{FEATURES}/bert.npy", mmap_mode="r").shape[0], "row count mismatch"
print(meta["split"].value_counts().to_string())

split
train    2736
test      588
val       586


## Cell 5: run the tests

In [5]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_bootstrap.py", "tests/test_confound_controls.py", "tests/test_evidence_eval.py"],
    cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1500:])
r.check_returncode()
print("Tests passed")

.....................                                                    [100%]
21 passed in 2.12s

Tests passed


## Cell 6: controls (script 10), with output shown

In [6]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/10_confound_controls.py"],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
r.check_returncode()

PROBS_DIR = f"{BFND}/results/controls/probs"
print("\nSaved control probability files:", len(os.listdir(PROBS_DIR)) if os.path.isdir(PROBS_DIR) else 0)

text_banglabert          seed 42: val macro-F1 0.6908
text_banglabert          seed 43: val macro-F1 0.6983
text_banglabert          seed 44: val macro-F1 0.6860
presence_only            seed 42: val macro-F1 0.1789
presence_only            seed 43: val macro-F1 0.1789
presence_only            seed 44: val macro-F1 0.1789
length_only              seed 42: val macro-F1 0.4557
length_only              seed 43: val macro-F1 0.4662
length_only              seed 44: val macro-F1 0.4349
text_plus_presence       seed 42: val macro-F1 0.6936
text_plus_presence       seed 43: val macro-F1 0.7089
text_plus_presence       seed 44: val macro-F1 0.6997
fusion_real              seed 42: val macro-F1 0.7297
fusion_real              seed 43: val macro-F1 0.7613
fusion_real              seed 44: val macro-F1 0.7427
fusion_image_shuffled    seed 42: val macro-F1 0.6874
fusion_image_shuffled    seed 43: val macro-F1 0.6869
fusion_image_shuffled    seed 44: val macro-F1 0.6689
# Confound controls (diagnos

## Cell 7: three paired bootstrap comparisons on the test split

The names come from the controls run. All three were fixed in advance (D-10).

In [7]:
import json, subprocess, sys

PROBS_DIR = f"{BFND}/results/controls/probs"
OUT_DIR = f"{BFND}/results/controls"
COMPARISONS = [
    # script 12 reports b minus a, so (a, b) = (baseline, comparison)
    ("text_plus_presence", "fusion_real"),        # fusion_real minus text_plus_presence: image content beyond presence
    ("fusion_image_shuffled", "fusion_real"),     # fusion_real minus fusion_image_shuffled: real pairing vs broken pairing
    ("text_banglabert", "text_plus_presence"),    # text_plus_presence minus text_banglabert: presence alone, on text
]

results = {}
for a, b in COMPARISONS:
    out = f"{OUT_DIR}/bootstrap_{b}_minus_{a}.json"
    r = subprocess.run(
        [sys.executable, "scripts/12_bootstrap_ci.py", "--a", a, "--b", b,
         "--probs-dir", PROBS_DIR, "--out", out],
        cwd=PROJECT, capture_output=True, text=True)
    print(f"=== {b} minus {a} ===")
    print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr[-2000:])
    r.check_returncode()
    results[(a, b)] = json.load(open(out))

=== fusion_real minus text_plus_presence ===
Macro-F1 fusion_real - text_plus_presence: +0.0752  95% CI [+0.0236, +0.1258]
Satire F1 fusion_real - text_plus_presence: +0.1362  95% CI [+0.0015, +0.2737]  (n_satire=32)
=== fusion_real minus fusion_image_shuffled ===
Macro-F1 fusion_real - fusion_image_shuffled: +0.0759  95% CI [+0.0174, +0.1324]
Satire F1 fusion_real - fusion_image_shuffled: +0.1527  95% CI [-0.0097, +0.3081]  (n_satire=32)
=== text_plus_presence minus text_banglabert ===
Macro-F1 text_plus_presence - text_banglabert: +0.0294  95% CI [-0.0103, +0.0740]
Satire F1 text_plus_presence - text_banglabert: +0.0815  95% CI [-0.0308, +0.2054]  (n_satire=32)


## Cell 8: summary table

In [8]:
rows = []
for (a, b), d in results.items():
    m, s = d["macro_f1"], d["satire_f1"]
    rows.append({
        "comparison": f"{b} minus {a}",
        "macro-F1 diff": round(m["diff_b_minus_a"], 4),
        "macro-F1 95% CI": f"[{m['ci_low']:+.4f}, {m['ci_high']:+.4f}]",
        "Satire F1 diff": round(s["diff_b_minus_a"], 4),
        "Satire F1 95% CI": f"[{s['ci_low']:+.4f}, {s['ci_high']:+.4f}]",
    })
import pandas as pd
pd.set_option("display.width", 200)
print(pd.DataFrame(rows).to_string(index=False))
print("\nReading: a CI that excludes 0 means the difference holds beyond resampling noise on the test rows.")

                              comparison  macro-F1 diff    macro-F1 95% CI  Satire F1 diff   Satire F1 95% CI
    fusion_real minus text_plus_presence         0.0752 [+0.0236, +0.1258]          0.1362 [+0.0015, +0.2737]
 fusion_real minus fusion_image_shuffled         0.0759 [+0.0174, +0.1324]          0.1527 [-0.0097, +0.3081]
text_plus_presence minus text_banglabert         0.0294 [-0.0103, +0.0740]          0.0815 [-0.0308, +0.2054]

Reading: a CI that excludes 0 means the difference holds beyond resampling noise on the test rows.


## Cell 9: copy results to Drive (copy only, nothing deleted)

In [9]:
import os, shutil, time
stamp = time.strftime("%Y%m%d_%H%M")
EXPORT = f"{BFND}/exports/step3_{stamp}"
os.makedirs(EXPORT, exist_ok=True)
src = f"{BFND}/results/controls"
shutil.copytree(src, f"{EXPORT}/controls", dirs_exist_ok=True)
print("Exported to:", f"{EXPORT}/controls")

Exported to: /content/drive/MyDrive/BFND-data/exports/step3_20261125_1827/controls
